In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight



In [2]:
smote = SMOTE(sampling_strategy="auto", random_state=101, k_neighbors=5)

In [ ]:
X_train_fit_selected_scaled = joblib.load("artifacts/standard_scaler_bundle.joblib")["X_train_fit_selected_scaled"]
X_val_selected = joblib.load("artifacts/standard_scaler_bundle.joblib")["X_val_selected"]
X_test_selected   = joblib.load("artifacts/standard_scaler_bundle.joblib")["X_test_selected  "]

In [ ]:
X_train_smote, y_train_smote = smote.fit_resample(X_train_fit_selected_scaled, y_train_fit)

In [ ]:
print(X_train_smote.shape)
print(y_train_smote.shape)

In [ ]:
num_classes = len(np.unique(y))
print(num_classes)

In [ ]:
cw_arr = compute_class_weight(class_weight="balanced", classes=np.arange(num_classes), y=y_train_fit)
class_weights = {i: float(w) for i, w in enumerate(cw_arr)}
print(class_weights)

In [ ]:
n_features = X_train_smote.shape[1]
# captures the number of input columns from the post-SMOTE train array.
print(n_features)

In [ ]:
X_tr_seq = X_train_smote.to_numpy().reshape((-1, n_features, 1))
# reshapes the train matrix to (N, F, 1) so Conv1D can treat the 40 features as a 1d sequence with one channel
print(X_tr_seq.shape)
X_val_seq = X_val_scaled.to_numpy().reshape((-1, n_features, 1))
# reshapes the validation matrix the same way; val remains untouched by SMOTE to reflect real distribution.
print(X_val_seq.shape)
X_test_seq = X_test_scaled.to_numpy().reshape((-1, n_features, 1))
# reshapes the test matrix likewise; test is the final holdout set used once at the end.
print(X_test_seq.shape)

In [ ]:
y_tr_cat = keras.utils.to_categorical(y_train_smote, num_classes=num_classes)
y_val_cat = keras.utils.to_categorical(y_val, num_classes=num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes=num_classes)
# turns integer labels into one-hot vectors for categorical cross-entropy.

In [ ]:
inputs = keras.Input(shape=(n_features, 1))
x = layers.Conv1D(32, 5, padding="same")(inputs)
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)
x = layers.Dropout(0.2)(x)

x = layers.Conv1D(64, 5, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)
x = layers.Dropout(0.2)(x)

x = layers.Conv1D(128, 3, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)
x = layers.Dropout(0.2)(x)

x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)

logits = layers.Dense(num_classes, activation=None)(x)
model = keras.Model(inputs=inputs, outputs=logits)


In [ ]:
loss = keras.losses.CategoricalCrossentropy(from_logits=True)
optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss=loss, metrics=[keras.metrics.CategoricalAccuracy(name="acc")])


In [ ]:
ckpt_path = "artifacts/models/cnn_model.keras"
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
ckpt = keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True)


In [ ]:
history = model.fit(
    X_tr_seq, y_tr_cat,
    validation_data=(X_val_seq, y_val_cat),
    epochs=20,
    batch_size=256,
    callbacks=[early_stop, ckpt],
    class_weight=class_weights,
    verbose=1
)


In [ ]:
best_model = keras.models.load_model(ckpt_path)
val_loss, val_acc = best_model.evaluate(X_val_seq, y_val_cat, batch_size=512, verbose=0)
test_loss, test_acc = best_model.evaluate(X_test_seq, y_test_cat, batch_size=512, verbose=0)

print({"val_loss": val_loss, "val_acc": val_acc, "test_loss": test_loss, "test_acc": test_acc})
